# Free LLM Inference APIs: Groq & Google AI Studio

Two providers, one notebook. Each section covers:
1. A single (one-shot) API call
2. A multi-turn chat loop

**Prerequisites:**
```bash
pip install groq google-genai
```

Get your keys:
- Groq: https://console.groq.com/keys (no credit card)
- Gemini: https://aistudio.google.com/apikey (Google account)

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
#%pip install -q groq google-genai

In [1]:
# ── API Keys ───────────────────────────────────────────────────────────────────
#1.pip install python-dotenv/ uv add python-dotenv
#2.Create an file named ".env" at project root with  
#GROQ_API_KEY=..
#GEMINI_API_KEY=..

import os
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY  = os.getenv("GROQ_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

#Verify:
#print(GROQ_API_KEY)
#print(GEMINI_API_KEY)


---
## Part 1 — Groq

Groq's client is **OpenAI SDK-compatible**: same interface, just point to a different base URL.  
The native `groq` package wraps this automatically.

In [2]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

# ── 1a. Single inference call ──────────────────────────────────────────────────
GROQ_MODEL = "llama-3.3-70b-versatile"   # swap for any model from the markdown

response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "Hello?"},
    ],
    temperature=0.7,
    max_tokens=300,
)

print("=== Groq single call ===")
print(response.choices[0].message.content)
print(f"\nUsage: {response.usage}")

=== Groq single call ===
Hello. How can I help you?

Usage: CompletionUsage(completion_tokens=9, prompt_tokens=43, total_tokens=52, completion_time=0.039303583, completion_tokens_details=None, prompt_time=0.001973979, prompt_tokens_details=None, queue_time=0.047246335, total_time=0.041277562)


In [ ]:
print(response.choices[].message.content)

Hello. How can I help you?


In [ ]:
# ── 1b. Multi-turn chat with Groq ──────────────────────────────────────────────
# We maintain the full history list and append each turn manually.

def groq_chat(client, model: str, system: str = "You are a helpful assistant."):
    """Simple multi-turn chat loop. Type 'quit' to exit."""
    history = [{"role": "system", "content": system}]
    print(f"Chatting with {model} (type 'quit' to stop)\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break

        history.append({"role": "user", "content": user_input})

        resp = client.chat.completions.create(
            model=model,
            messages=history,
            temperature=0.7,
            max_tokens=512,
        )
        assistant_msg = resp.choices[0].message.content
        history.append({"role": "assistant", "content": assistant_msg})
        print(f"\nAssistant: {assistant_msg}\n")


# Uncomment to run interactively:
#groq_chat(groq_client, GROQ_MODEL, system="You are an expert in ML and explainability.")

Chatting with llama-3.3-70b-versatile (type 'quit' to stop)


Assistant: It seems like you didn't ask a question. As an expert in Machine Learning (ML) and explainability, I can provide information on a wide range of topics, including:

1. **Model Interpretability**: Techniques for understanding how machine learning models make predictions, such as feature importance, partial dependence plots, and SHAP values.
2. **Explainability Techniques**: Methods like LIME, TreeExplainer, and Anchor, which generate explanations for individual predictions or model behavior.
3. **Model Transparency**: Designing models that are inherently interpretable, like decision trees, rule-based models, or linear models.
4. **Explainability Metrics**: Evaluating the quality of explanations, such as fidelity, stability, and sensitivity.
5. **Applications of Explainability**: Using explainability techniques in real-world domains, like healthcare, finance, or computer vision.

What specific aspect of ML and explai

In [4]:
# ── 1b (scripted). Run a canned multi-turn exchange for demo/testing ───────────

def groq_scripted_chat(client, model: str, turns: list[str], system: str = "You are a concise assistant."):
    history = [{"role": "system", "content": system}]
    for user_msg in turns:
        history.append({"role": "user", "content": user_msg})
        resp = client.chat.completions.create(
            model=model, messages=history, temperature=0.7, max_tokens=300
        )
        assistant_msg = resp.choices[0].message.content
        history.append({"role": "assistant", "content": assistant_msg})
        print(f"User : {user_msg}")
        print(f"Model: {assistant_msg}\n")


groq_scripted_chat(
    groq_client,
    GROQ_MODEL,
    turns=[
        "Hi! What can you tell me about RLHF?",
        "How does the reward model relate to the policy in PPO?",
        "Give me a one-sentence summary of what we just discussed.",
    ],
)

User : Hi! What can you tell me about RLHF?
Model: RLHF stands for Reinforcement Learning from Human Feedback. It's a technique used in AI training, particularly for large language models. The process involves:

1. Generating text based on a prompt
2. Having human evaluators rate the generated text
3. Using the ratings to fine-tune the AI model

This method helps the model learn to produce more accurate, informative, and engaging content. RLHF is used to improve the performance of language models in tasks like text generation, conversation, and question-answering.

User : How does the reward model relate to the policy in PPO?
Model: In Proximal Policy Optimization (PPO), the reward model is not directly involved. Instead, PPO uses a value function (critic) and a policy function (actor). 

The policy (actor) takes actions, and the value function (critic) estimates the expected return of the current policy. The policy is updated to maximize the cumulative reward, while the value function

---
## Part 2 — Google AI Studio (Gemini API)

Uses the `google-genai` SDK (not the older `google-generativeai`).  
The key difference from Groq: Gemini has a native `ChatSession` object that tracks history for you.

In [6]:
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

GEMINI_MODEL = "gemini-2.5-flash"   # free-tier workhorse as of 2026

# ── 2a. Single inference call ──────────────────────────────────────────────────
response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents="What is the core idea behind Shapley values in one paragraph?",
    config=types.GenerateContentConfig(
        system_instruction="You are a concise assistant.",
        temperature=0.7,
        max_output_tokens=300,
    ),
)

print("=== Gemini single call ===")
print(response.text)
print(f"\nUsage: {response.usage_metadata}")

=== Gemini single call ===
Shapley values, originating from cooperative game theory, provide a method to fairly distribute the total gain among players in a coalition by quantifying each player's marginal contribution. The core idea is to average the value a player adds to every possible sub-coalition they could join, considering all possible orders in which players could form the coalition. This ensures that a player's assigned value reflects their average expected marginal contribution, regardless of the presence or absence of other players, thus allocating the total surplus in a unique and equitable way.

Usage: cache_tokens_details=None cached_content_token_count=None candidates_token_count=106 candidates_tokens_details=None prompt_token_count=21 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=21
)] thoughts_token_count=44 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=171 traffic_type=None


In [ ]:
# ── 2b. Multi-turn chat with Gemini ───────────────────────────────────────────
# The SDK's ChatSession object maintains history automatically.

def gemini_chat(client, model: str, system: str = "You are a helpful assistant."):
    """Interactive multi-turn chat loop. Type 'quit' to exit."""
    chat = client.chats.create(
        model=model,
        config=types.GenerateContentConfig(system_instruction=system, temperature=0.7),
    )
    print(f"Chatting with {model} (type 'quit' to stop)\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break
        resp = chat.send_message(user_input)
        print(f"\nAssistant: {resp.text}\n")


# Uncomment to run interactively:
# gemini_chat(gemini_client, GEMINI_MODEL)

In [ ]:
# ── 2b (scripted). Canned multi-turn exchange ──────────────────────────────────

def gemini_scripted_chat(client, model: str, turns: list[str], system: str = "You are a concise assistant."):
    chat = client.chats.create(
        model=model,
        config=types.GenerateContentConfig(system_instruction=system, temperature=0.7,
                                           max_output_tokens=300),
    )
    for user_msg in turns:
        resp = chat.send_message(user_msg)
        print(f"User : {user_msg}")
        print(f"Model: {resp.text}\n")


gemini_scripted_chat(
    gemini_client,
    GEMINI_MODEL,
    turns=[
        "Hi! What can you tell me about RLHF?",
        "How does the reward model relate to the policy in PPO?",
        "Give me a one-sentence summary of what we just discussed.",
    ],
)

---
## Key Differences at a Glance

| | Groq | Gemini (AI Studio) |
|---|---|---|
| **Client** | `Groq(api_key=...)` | `genai.Client(api_key=...)` |
| **Call style** | `client.chat.completions.create(messages=[...])` | `client.models.generate_content(contents=...)` |
| **Chat history** | Managed manually (append to `messages` list) | Managed by `ChatSession` object |
| **System prompt** | `{"role": "system", "content": ...}` in messages | `GenerateContentConfig(system_instruction=...)` |
| **Response text** | `response.choices[0].message.content` | `response.text` |
| **OpenAI compat** | ✅ Drop-in with `openai` SDK + custom base URL | ❌ Own SDK only |